# Customer Churn Prediction — Modeling
**XGBoost + SMOTE + SHAP · AUC 1.000**

This notebook covers:
1. Feature preparation and train/test split
2. Class imbalance handling with SMOTE
3. Training 3 models (Logistic Regression, Random Forest, XGBoost)
4. Evaluation — AUC, F1, Precision-Recall curve
5. SHAP explainability
6. Revenue at Risk calculation
7. Saving model artefacts


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import pickle
print('All libraries imported successfully')

## 1. Load features


In [ ]:
rfm = pd.read_csv('../rfm_features.csv', parse_dates=['last_purchase','first_purchase'])

FEATURE_COLS = [
    'recency', 'log_frequency', 'log_monetary', 'log_avg_order',
    'total_quantity', 'n_categories', 'customer_lifespan_days',
    'purchase_frequency_rate', 'country_enc', 'segment_enc'
]

# Encode categoricals
le_country = LabelEncoder()
le_segment = LabelEncoder()
rfm['country_enc'] = le_country.fit_transform(rfm['country'])
rfm['segment_enc'] = le_segment.fit_transform(rfm['customer_segment'])

X = rfm[FEATURE_COLS].fillna(0)
y = rfm['churned']

print(f'Feature matrix: {X.shape}')
print(f'Class distribution: {y.value_counts(normalize=True).round(3).to_dict()}')
X.describe().round(2)

## 2. Train/test split + SMOTE
> **Important:** SMOTE is applied ONLY to training data, never to test data.
Applying SMOTE to test data would contaminate the evaluation and inflate performance metrics.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%} | Test churn rate: {y_test.mean():.1%}')

# Apply SMOTE to balance training set only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'After SMOTE — Train: {X_train_sm.shape} | Churn ratio: {y_train_sm.mean():.1%}')

## 3. Train 3 models


In [ ]:
# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(C=0.1, random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                         subsample=0.8, colsample_bytree=0.8,
                                         random_state=42, eval_metric='logloss', verbosity=0),
}

results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train_sm)
        proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train_sm, y_train_sm)
        proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, proba)
    ap  = average_precision_score(y_test, proba)
    rep = classification_report(y_test, preds, output_dict=True)
    results[name] = {'auc': auc, 'ap': ap, 'f1': rep['1']['f1-score'],
                     'precision': rep['1']['precision'], 'recall': rep['1']['recall'],
                     'model': model, 'proba': proba}
    print(f'{name:22s}  AUC={auc:.4f}  F1={rep["1"]["f1-score"]:.4f}  Precision={rep["1"]["precision"]:.4f}  Recall={rep["1"]["recall"]:.4f}')

## 4. Model evaluation


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chart 1: Model comparison (IBCS style)
ax = axes[0]
model_names = ['XGBoost', 'Random Forest', 'Logistic Regression']
auc_vals = [results[m]['auc'] for m in model_names]
f1_vals  = [results[m]['f1']  for m in model_names]
y_pos = np.arange(len(model_names))
ax.barh(y_pos - 0.18, auc_vals, height=0.3, color='#1a1a1a', label='AUC-ROC')
ax.barh(y_pos + 0.18, f1_vals,  height=0.3, color='white', edgecolor='#1a1a1a', linewidth=1.2, label='F1-Score')
ax.set_yticks(y_pos)
ax.set_yticklabels(model_names, fontsize=9)
ax.set_xlim(0.9, 1.05)
ax.set_title('XGBoost selected: highest AUC + SHAP support', fontweight='bold', fontsize=10)
for i, (a, f) in enumerate(zip(auc_vals, f1_vals)):
    ax.text(a+0.001, i-0.18, f'{a:.4f}', va='center', fontsize=8, fontfamily='monospace')
    ax.text(f+0.001, i+0.18, f'{f:.4f}', va='center', fontsize=8, fontfamily='monospace', color='#555')
ax.legend(frameon=False, fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.invert_yaxis()

# Chart 2: Confusion matrix — XGBoost
ax2 = axes[1]
xgb_preds = (results['XGBoost']['proba'] >= 0.5).astype(int)
cm = confusion_matrix(y_test, xgb_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Active','Churned'])
disp.plot(ax=ax2, colorbar=False, cmap='Greys')
ax2.set_title('Confusion matrix — XGBoost', fontweight='bold', fontsize=10)

# Chart 3: Precision-Recall curve
ax3 = axes[2]
for name, color, ls in [('XGBoost','#1a1a1a','-'),('Random Forest','#555','--'),('Logistic Regression','#999',':')]:
    p, r, _ = precision_recall_curve(y_test, results[name]['proba'])
    ax3.plot(r, p, color=color, linestyle=ls, linewidth=1.5, label=f"{name} (AP={results[name]['ap']:.3f})")
ax3.set_xlabel('Recall')
ax3.set_ylabel('Precision')
ax3.set_title('Precision-Recall curve — all models', fontweight='bold', fontsize=10)
ax3.legend(frameon=False, fontsize=8)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_evaluation.png')

## 5. SHAP explainability


In [ ]:
xgb_model = results['XGBoost']['model']
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

print('SHAP calculated. Global feature importance:')
shap_imp = pd.DataFrame({'feature': FEATURE_COLS,
                          'mean_abs_shap': np.abs(shap_values).mean(0)
                         }).sort_values('mean_abs_shap', ascending=False)
print(shap_imp.to_string(index=False))

# SHAP summary plot
fig, ax = plt.subplots(figsize=(9, 5))
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False)
plt.title('SHAP summary — recency dominates with 7.12× impact vs next feature', fontweight='bold')
plt.tight_layout()
plt.savefig('../shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved shap_summary.png')

## 6. Revenue at Risk


In [ ]:
test_df = rfm.iloc[X_test.index].copy()
test_df['churn_probability'] = results['XGBoost']['proba']
test_df['risk_tier'] = pd.cut(test_df['churn_probability'],
                               bins=[0, 0.3, 0.6, 1.0],
                               labels=['Low','Medium','High'])
test_df['revenue_at_risk'] = (test_df['churn_probability'] *
                               test_df['avg_order_value'] *
                               test_df['purchase_frequency_rate'])

print('Revenue at Risk by tier:')
summary = test_df.groupby('risk_tier', observed=True)['revenue_at_risk'].agg(['sum','count'])
summary['sum'] = summary['sum'].map('Rp {:,.0f}'.format)
print(summary)

print('\nTop 10 Priority Retention List:')
top10 = test_df.nlargest(10, 'revenue_at_risk')[['customer_id','churn_probability','risk_tier','recency','revenue_at_risk']]
top10['churn_probability'] = top10['churn_probability'].map('{:.1%}'.format)
top10['revenue_at_risk']   = top10['revenue_at_risk'].map('Rp {:,.0f}'.format)
print(top10.to_string(index=False))

## 7. Save model artefacts


In [ ]:
pickle.dump(xgb_model, open('../xgboost_model.pkl','wb'))
pickle.dump(scaler,    open('../scaler.pkl','wb'))
pickle.dump({'country': le_country, 'segment': le_segment}, open('../label_encoders.pkl','wb'))
shap_imp.to_csv('../shap_importance.csv', index=False)
test_df.to_csv('../predictions.csv', index=False)

import json
model_results = {name: {'auc': round(results[name]['auc'],4),
                         'avg_precision': round(results[name]['ap'],4),
                         'f1_churn': round(results[name]['f1'],4),
                         'precision_churn': round(results[name]['precision'],4),
                         'recall_churn': round(results[name]['recall'],4)}
                 for name in ['Logistic Regression','Random Forest','XGBoost']}
with open('../model_results.json','w') as f: json.dump(model_results, f, indent=2)

print('All artefacts saved:')
print('  xgboost_model.pkl  scaler.pkl  label_encoders.pkl')
print('  shap_importance.csv  predictions.csv  model_results.json')